# Spatiotemporal Wildfire Prediction — EDA

**Competition:** https://www.kaggle.com/competitions/spatiotemporal-wildfire-prediction  
**Target:** `y` — binary (1 = wildfire occurred in this window, 0 = no fire)  
**Structure:** Each `window_id` = a lat/lon location. 60 daily rows of weather/fire-index features per window.  
Labels are per-window (not per-day), so we predict probability of fire for each 60-day window.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy.stats import skew, kurtosis
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

DATA = '/Users/vkdvamshi/gitRepos/ai-ml-learnings/wildFires -  ML/'

## 1. Load & Merge Data

In [ ]:
train_feat   = pd.read_csv(DATA + 'train_features.csv')
train_labels = pd.read_csv(DATA + 'train_labels.csv')
test_feat    = pd.read_csv(DATA + 'test_features.csv')

# Merge features with labels
train = pd.merge(train_feat, train_labels, on='window_id', how='inner')

# Parse lat / lon / window_number from window_id
for df in [train, test_feat]:
    parsed = df['window_id'].str.extract(r'([\d.\-]+)_([\d.\-]+)_(\d+)')
    df['lat']     = parsed[0].astype(float)
    df['lon']     = parsed[1].astype(float)
    df['win_num'] = parsed[2].astype(int)

print(f'Train rows : {len(train):,}  |  Unique windows: {train["window_id"].nunique():,}')
print(f'Test  rows : {len(test_feat):,}  |  Unique windows: {test_feat["window_id"].nunique():,}')
print(f'Days per window: {train.groupby("window_id")["day"].count().unique()}  (all windows have same length)')
train.head(3)

## 2. Data Quality Check

In [ ]:
print('=== dtypes ===')
print(train.dtypes)
print()
print('=== Missing Values ===')
missing = train.isnull().sum()
print(missing[missing > 0] if missing.any() else 'No missing values')
print()
print('=== Duplicate window_id rows ===')
dup_windows = train_labels['window_id'].duplicated().sum()
print(f'{dup_windows} duplicate window_ids in labels')

In [ ]:
train.describe().T.style.background_gradient(cmap='Blues', subset=['mean','std','min','max'])

## 3. Target Variable — Class Balance

In [ ]:
label_counts = train_labels['y'].value_counts()
pos_rate = train_labels['y'].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(['No Fire (0)', 'Fire (1)'], label_counts.values, color=['steelblue', 'tomato'], edgecolor='white')
axes[0].set_title('Class Distribution (per window)')
axes[0].set_ylabel('Count')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 200, f'{v:,}\n({v/len(train_labels)*100:.1f}%)', ha='center', fontsize=11)

# Pie chart
axes[1].pie(label_counts.values, labels=['No Fire', 'Fire'], autopct='%1.1f%%',
            colors=['steelblue', 'tomato'], startangle=90, wedgeprops={'edgecolor': 'white'})
axes[1].set_title(f'Fire Rate: {pos_rate:.2%} positive')

plt.suptitle('Target Variable: Wildfire Occurrence per 60-day Window', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f'\nClass imbalance ratio (no-fire : fire) = {label_counts[0]/label_counts[1]:.1f} : 1')
print('Moderate imbalance — consider class_weight="balanced" or scale_pos_weight in tree models')

## 4. Feature Glossary

| Feature | Description | Unit |
|---------|-------------|------|
| `pr` | Precipitation | mm |
| `rmax` | Max relative humidity | % |
| `rmin` | Min relative humidity | % |
| `sph` | Specific humidity | kg/kg |
| `srad` | Downward shortwave solar radiation | W/m² |
| `tmmn` | Min air temperature | K |
| `tmmx` | Max air temperature | K |
| `vs` | Wind speed (10m) | m/s |
| `bi` | Burning Index (fire behavior) | NFDRS |
| `fm100` | 100-hr dead fuel moisture | % |
| `fm1000` | 1000-hr dead fuel moisture | % |
| `erc` | Energy Release Component | NFDRS |
| `etr` | Reference evapotranspiration | mm |
| `pet` | Potential evapotranspiration | mm |
| `vpd` | Vapor Pressure Deficit | kPa |
| `day` | Day within the 60-day window | 1–60 |

## 5. Feature Distributions

In [ ]:
feat_cols = [c for c in train.columns if c not in ('window_id', 'y', 'lat', 'lon', 'win_num')]

# Compute skewness
skew_vals = train[feat_cols].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
print('Feature Skewness (sorted):')
print(skew_vals.to_frame('skew').style.bar(color=['tomato','steelblue'], align='zero'))

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(feat_cols):
    ax = axes[i]
    # Split by fire/no-fire for comparison
    no_fire = train.loc[train['y'] == 0, col].sample(min(50000, (train['y']==0).sum()), random_state=42)
    fire    = train.loc[train['y'] == 1, col].sample(min(50000, (train['y']==1).sum()), random_state=42)
    ax.hist(no_fire, bins=40, alpha=0.5, color='steelblue', label='No Fire', density=True)
    ax.hist(fire,    bins=40, alpha=0.5, color='tomato',    label='Fire',    density=True)
    ax.set_title(f'{col}  (skew={skew_vals[col]:.2f})', fontsize=9)
    ax.set_yticks([])
    if i == 0:
        ax.legend(fontsize=8)

# Hide unused subplots
for j in range(len(feat_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions: Fire vs No-Fire Windows (density, 50k sample)', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Window-Level Aggregations — Fire vs No-Fire

Since labels are per-window, aggregate features to window level before comparing.

In [ ]:
# Aggregate to window level (mean of each 60-day window)
agg_cols = [c for c in feat_cols if c != 'day']
window_df = train.groupby('window_id')[agg_cols].mean()
window_df['y'] = train.groupby('window_id')['y'].first()
window_df['lat'] = train.groupby('window_id')['lat'].first()
window_df['lon'] = train.groupby('window_id')['lon'].first()

print(f'Window-level dataframe: {window_df.shape}')
window_df.head(6)

In [ ]:
fire_indices = ['erc', 'bi', 'vpd', 'vs', 'fm100', 'fm1000', 'tmmx', 'pr', 'rmin', 'sph']

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(fire_indices):
    ax = axes[i]
    data_0 = window_df.loc[window_df['y'] == 0, col]
    data_1 = window_df.loc[window_df['y'] == 1, col]
    ax.violinplot([data_0, data_1], positions=[0, 1], showmedians=True)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['No Fire', 'Fire'])
    ax.set_title(col, fontsize=11)

plt.suptitle('Window-Level Feature Distributions: Fire vs No-Fire (violin plots)', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Correlation Matrix

In [ ]:
corr = window_df[agg_cols + ['y']].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.3,
            annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix (window-level means)', fontsize=13)
plt.tight_layout()
plt.show()

# Highly correlated pairs
upper = corr.where(mask == False).abs()
high_corr = [(r, c, corr.loc[r,c]) for r in corr.index for c in corr.columns
             if r != c and upper.loc[r,c] > 0.85]
high_corr = sorted(set([tuple(sorted((r,c))) + (v,) for r,c,v in high_corr]), key=lambda x: abs(x[2]), reverse=True)
print('Highly correlated pairs (|r| > 0.85):')
for r, c, v in high_corr:
    print(f'  {r:10s} <-> {c:10s}  r={v:.3f}')

## 8. Correlation with Target

In [ ]:
target_corr = corr['y'].drop('y').sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['tomato' if v > 0 else 'steelblue' for v in target_corr]
target_corr.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Target (y)', fontsize=13)
ax.set_xlabel('Pearson r')
for bar, val in zip(ax.patches, target_corr):
    ax.text(val + (0.005 if val >= 0 else -0.005), bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)
plt.tight_layout()
plt.show()

print('Top positive correlators (fire risk ↑):', target_corr[target_corr > 0].tail(5).index.tolist())
print('Top negative correlators (fire risk ↓):', target_corr[target_corr < 0].head(5).index.tolist())

## 9. Temporal Analysis — Feature Trends Over 60 Days

In [ ]:
# Mean feature value per day, split by fire/no-fire
daily = train.groupby(['day', 'y'])[['erc', 'bi', 'vpd', 'vs', 'tmmx', 'pr']].mean().reset_index()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
plot_cols = ['erc', 'bi', 'vpd', 'vs', 'tmmx', 'pr']

for i, col in enumerate(plot_cols):
    ax = axes[i]
    for label, color, lname in [(0, 'steelblue', 'No Fire'), (1, 'tomato', 'Fire')]:
        d = daily[daily['y'] == label]
        ax.plot(d['day'], d[col], color=color, label=lname, linewidth=2)
    ax.set_title(col, fontsize=11)
    ax.set_xlabel('Day in Window')
    if i == 0:
        ax.legend()

plt.suptitle('Mean Feature Values per Day: Fire vs No-Fire Windows', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Geospatial Distribution of Fire vs No-Fire Windows

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, label, color, title in [
    (axes[0], 0, 'steelblue', 'No Fire Windows'),
    (axes[1], 1, 'tomato',    'Fire Windows')
]:
    subset = window_df[window_df['y'] == label]
    sc = ax.scatter(subset['lon'], subset['lat'], s=2, alpha=0.4, c=color)
    ax.set_title(f'{title} (n={len(subset):,})', fontsize=12)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_xlim(-130, -60)
    ax.set_ylim(24, 50)

plt.suptitle('Geographic Distribution of Fire vs No-Fire Windows (CONUS)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Fire rate by latitude band
window_df['lat_band'] = pd.cut(window_df['lat'], bins=np.arange(24, 51, 2), right=False)
fire_by_lat = window_df.groupby('lat_band')['y'].agg(['mean', 'count']).reset_index()
fire_by_lat.columns = ['lat_band', 'fire_rate', 'count']

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(fire_by_lat)), fire_by_lat['fire_rate'], color='tomato', edgecolor='white')
ax.set_xticks(range(len(fire_by_lat)))
ax.set_xticklabels([str(b) for b in fire_by_lat['lat_band']], rotation=45, ha='right')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title('Wildfire Rate by Latitude Band')
ax.set_ylabel('Fire Rate')
plt.tight_layout()
plt.show()

## 11. Skewed Features — Log Transform Effect

In [ ]:
high_skew = skew_vals[skew_vals.abs() > 0.75].index.tolist()
print(f'Features with |skew| > 0.75: {high_skew}')

fig, axes = plt.subplots(len(high_skew), 2, figsize=(12, 3 * len(high_skew)))
if len(high_skew) == 1:
    axes = axes.reshape(1, -1)

sample = train.sample(100000, random_state=42)
for i, col in enumerate(high_skew):
    axes[i, 0].hist(sample[col].dropna(), bins=50, color='steelblue', edgecolor='white')
    axes[i, 0].set_title(f'{col}  (skew={skew_vals[col]:.2f})')
    axes[i, 1].hist(np.log1p(sample[col].clip(lower=0).dropna()), bins=50, color='seagreen', edgecolor='white')
    axes[i, 1].set_title(f'log1p({col})  (skew={skew(np.log1p(sample[col].clip(lower=0).dropna())):.2f})')

plt.suptitle('Skewed Features: Raw vs log1p Transformed', fontsize=13)
plt.tight_layout()
plt.show()

## 12. Key Fire Weather Index: ERC & BI Deep Dive

In [ ]:
# ERC and BI are the strongest fire risk indicators
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col in zip(axes, ['erc', 'bi']):
    # Bin the index and compute fire rate
    bins = pd.cut(window_df[col], bins=20)
    fire_rate = window_df.groupby(bins)['y'].mean()
    count = window_df.groupby(bins)['y'].count()
    
    ax2 = ax.twinx()
    ax2.bar(range(len(fire_rate)), count.values, color='lightgray', alpha=0.5, label='Count')
    ax.plot(range(len(fire_rate)), fire_rate.values, color='tomato', marker='o', ms=5, label='Fire Rate')
    ax.set_title(f'{col.upper()} vs Fire Rate', fontsize=12)
    ax.set_xlabel(f'{col.upper()} bin')
    ax.set_ylabel('Fire Rate', color='tomato')
    ax2.set_ylabel('Window Count', color='gray')
    ax.set_xticks(range(len(fire_rate)))
    ax.set_xticklabels([f'{b.mid:.0f}' for b in fire_rate.index], rotation=45, ha='right', fontsize=7)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

plt.suptitle('Fire Rate by ERC and Burning Index (window-level means)', fontsize=13)
plt.tight_layout()
plt.show()

## 13. Feature Importance Hint — Point-Biserial Correlation

In [ ]:
from scipy.stats import pointbiserialr

results = {}
for col in agg_cols:
    r, p = pointbiserialr(window_df['y'], window_df[col])
    results[col] = {'r': r, 'p': p}

pb_df = pd.DataFrame(results).T.sort_values('r')
pb_df['significant'] = pb_df['p'] < 0.001

fig, ax = plt.subplots(figsize=(8, 7))
colors = ['tomato' if v > 0 else 'steelblue' for v in pb_df['r']]
ax.barh(pb_df.index, pb_df['r'], color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Point-Biserial Correlation: Feature vs Wildfire (y)', fontsize=13)
ax.set_xlabel('r  (all p < 0.001)')
plt.tight_layout()
plt.show()

print('All features are statistically significant (p < 0.001)')
print(pb_df[['r']].round(3))

## 14. Train vs Test Feature Distribution Shift

In [ ]:
# Check if test features have different distributions (domain shift)
check_cols = ['erc', 'bi', 'vpd', 'tmmx', 'pr', 'sph']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

train_sample = train.sample(min(200000, len(train)), random_state=42)
test_sample  = test_feat.sample(min(200000, len(test_feat)), random_state=42)

for i, col in enumerate(check_cols):
    ax = axes[i]
    ax.hist(train_sample[col], bins=40, alpha=0.5, density=True, color='steelblue', label='Train')
    ax.hist(test_sample[col],  bins=40, alpha=0.5, density=True, color='orange',    label='Test')
    ax.set_title(col, fontsize=11)
    ax.set_yticks([])
    if i == 0:
        ax.legend()

plt.suptitle('Train vs Test Feature Distributions (potential distribution shift)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
alldfs = [var for var in globals() if isinstance(globals()[var], pd.DataFrame)]
print(alldfs)

## 15. EDA Summary & Modeling Recommendations

### Key Findings

| Finding | Detail |
|---------|--------|
| **Class imbalance** | 26% fire rate — moderate; use `scale_pos_weight` or `class_weight` |
| **No missing values** | Clean dataset |
| **Window structure** | Each window = 60 days of daily weather data, 1 label per window |
| **Strongest fire signals** | `erc`, `bi`, `vpd`, `tmmx` positively correlated with fire |
| **Protective factors** | `fm100`, `fm1000`, `pr`, `rmax` negatively correlated |
| **Redundant features** | `tmmn`/`tmmx` highly correlated (~0.95+); `etr`/`pet` nearly identical |
| **Skewed features** | `pr`, `sph`, `vs`, `vpd` — apply `log1p` transform |
| **Geographic pattern** | Western US (lower longitudes) has higher fire rate |
| **Temporal pattern** | Fire windows show higher ERC/BI from day 1 — not just end-of-window spikes |

### Modeling Notes

1. **Frame as classification**, not regression — target is binary, use `log_loss` or `roc_auc`
2. **Feature engineering**: aggregate daily rows to window-level stats (mean, max, last_N_days)
3. **Consider temporal features**: slope/trend of ERC/BI/VPD over 60 days
4. **Drop `etr` or `pet`** (near-identical, pick one)
5. **Spatial features**: lat/lon as features (fire risk is geographically structured)
6. **Best starting model**: LightGBM or XGBoost with window-level aggregated features

# Wildfire Prediction — Model Selection

**Goal:** Identify the best model for binary wildfire prediction per 60-day window  
**Benchmark results (5-fold StratifiedKFold):**

| Model | ROC-AUC | Log-Loss |
|-------|---------|----------|
| Logistic Regression | 0.6569 | 0.6530 |
| Random Forest | **0.7470** | **0.4915** |
| Extra Trees | 0.7451 | 0.5051 |
| HistGradientBoosting | 0.7318 | 0.5009 |
| LightGBM | **0.7472** | 0.5469 |
| XGBoost | 0.7404 | 0.5648 |

**Winner:** LightGBM (AUC) / RandomForest (Log-Loss). We'll tune LightGBM + calibrate it.

In [ ]:
#train_feat', 'train_labels', 'test_feat', 'train', 'df', 'parsed', '_2', '_5', 
# 'window_df', '_11', 'corr', 'upper', 'daily', 'd', 'subset', 'fire_by_lat', 
# 'sample', 'pb_df', 'train_sample', 'test_sample', '_24', '_30']

window_df_sub = window_df.drop(columns=['pet','lat_band'])
window_df_sub

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy.stats import linregress
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                               HistGradientBoostingClassifier)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import roc_auc_score, log_loss, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

DATA = '/Users/vkdvamshi/gitRepos/ai-ml-learnings/wildFires -  ML/'
SEED = 42

## 1. Feature Engineering — Window-Level Aggregation

Each window = 60 daily rows → 1 prediction row.  
Features: mean, max, std, first-10-day avg, last-10-day avg, trend slope (linear regression over 60 days)

In [ ]:
train_feat   = pd.read_csv(DATA + 'train_features.csv')
train_labels = pd.read_csv(DATA + 'train_labels.csv')
test_feat    = pd.read_csv(DATA + 'test_features.csv')

train_raw = pd.merge(train_feat, train_labels, on='window_id', how='inner')

for df in [train_raw, test_feat]:
    parsed = df['window_id'].str.extract(r'([\d.\-]+)_([\d.\-]+)_(\d+)')
    df['lat']     = parsed[0].astype(float)
    df['lon']     = parsed[1].astype(float)
    df['win_num'] = parsed[2].astype(int)

daily_cols = [c for c in train_feat.columns if c not in ('window_id', 'day')]
trend_cols = ['erc', 'bi', 'vpd', 'tmmx', 'fm100', 'vs', 'pr']

def build_features(df):
    rows = []
    for wid, grp in df.groupby('window_id'):
        grp = grp.sort_values('day')
        row = {}
        for c in daily_cols:
            row[f'{c}_mean']    = grp[c].mean()
            row[f'{c}_max']     = grp[c].max()
            row[f'{c}_std']     = grp[c].std()
            row[f'{c}_last10']  = grp[c].tail(10).mean()
            row[f'{c}_first10'] = grp[c].head(10).mean()
        for c in trend_cols:
            slope, *_ = linregress(grp['day'], grp[c])
            row[f'{c}_slope'] = slope
        row['lat']     = grp['lat'].iloc[0]
        row['lon']     = grp['lon'].iloc[0]
        row['win_num'] = grp['win_num'].iloc[0]
        if 'y' in grp.columns:
            row['y'] = grp['y'].iloc[0]
        rows.append(row)
    return pd.DataFrame(rows)

print('Building train windows...')
window_train = build_features(train_raw)
print('Building test windows...')
window_test  = build_features(test_feat)

feat_cols = [c for c in window_train.columns if c != 'y']
X = window_train[feat_cols]
y = window_train['y']
X_sub = window_test[feat_cols]

print(f'Train: {X.shape}  |  Test: {X_sub.shape}  |  Positive rate: {y.mean():.3f}')

In [ ]:
X = window_df_sub.drop(columns=['y'])
y = window_df_sub['y']
train_feat

## 2. Benchmark — All Models (5-Fold Stratified CV)

In [ ]:

imbalance_ratio = (y == 0).sum() / (y == 1).sum()  # ~2.8
SEED = 42
models = {
    'LogisticReg':  make_pipeline(StandardScaler(),
                        LogisticRegression(max_iter=1000, class_weight='balanced', C=0.1)),
    'RandomForest': RandomForestClassifier(n_estimators=300, class_weight='balanced',
                        n_jobs=-1, random_state=SEED),
    'ExtraTrees':   ExtraTreesClassifier(n_estimators=300, class_weight='balanced',
                        n_jobs=-1, random_state=SEED),
    'HistGBT':      HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05,
                        max_depth=8, random_state=SEED),
    'LightGBM':     lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                        scale_pos_weight=imbalance_ratio, n_jobs=-1, random_state=SEED, verbosity=-1),
    'XGBoost':      xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                        scale_pos_weight=imbalance_ratio, eval_metric='logloss',
                        n_jobs=-1, random_state=SEED, verbosity=0),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
results = []

for name, model in models.items():
    scores = cross_validate(model, X, y, cv=cv,
                            scoring=['roc_auc', 'neg_log_loss'],
                            n_jobs=-1)
    results.append({
        'Model':    name,
        'ROC-AUC':  scores['test_roc_auc'].mean(),
        'AUC-Std':  scores['test_roc_auc'].std(),
        'Log-Loss': -scores['test_neg_log_loss'].mean(),
        'LL-Std':   scores['test_neg_log_loss'].std(),
    })
    print(f"{name:<15}  AUC={scores['test_roc_auc'].mean():.4f} ± {scores['test_roc_auc'].std():.4f}  "
          f"LL={-scores['test_neg_log_loss'].mean():.4f}")

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
display(results_df.set_index('Model').style.highlight_max(subset=['ROC-AUC'], color='lightgreen')
                                           .highlight_min(subset=['Log-Loss'], color='lightgreen')
                                           .format(precision=4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['tomato' if r == results_df['ROC-AUC'].max() else 'steelblue'
          for r in results_df['ROC-AUC']]
axes[0].barh(results_df['Model'], results_df['ROC-AUC'], color=colors, xerr=results_df['AUC-Std'],
             edgecolor='white', capsize=4)
axes[0].set_title('ROC-AUC (higher = better)', fontsize=12)
axes[0].axvline(0.5, color='gray', ls='--', lw=0.8)
for i, (v, e) in enumerate(zip(results_df['ROC-AUC'], results_df['AUC-Std'])):
    axes[0].text(v + e + 0.001, i, f'{v:.4f}', va='center', fontsize=9)

colors2 = ['tomato' if r == results_df['Log-Loss'].min() else 'steelblue'
           for r in results_df['Log-Loss']]
axes[1].barh(results_df['Model'], results_df['Log-Loss'], color=colors2, xerr=results_df['LL-Std'],
             edgecolor='white', capsize=4)
axes[1].set_title('Log-Loss (lower = better)', fontsize=12)
for i, (v, e) in enumerate(zip(results_df['Log-Loss'], results_df['LL-Std'])):
    axes[1].text(v + e + 0.001, i, f'{v:.4f}', va='center', fontsize=9)

plt.suptitle('Model Comparison — 5-Fold Stratified CV', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Feature Importance Analysis

In [ ]:
# Fit both top models on full data for feature importance
lgbm_full = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                                scale_pos_weight=imbalance_ratio, n_jobs=-1, random_state=SEED, verbosity=-1)
lgbm_full.fit(X, y)

rf_full = RandomForestClassifier(n_estimators=300, class_weight='balanced', n_jobs=-1, random_state=SEED)
rf_full.fit(X, y)

fi_lgbm = pd.Series(lgbm_full.feature_importances_, index=X.columns).sort_values(ascending=False).head(25)
fi_rf   = pd.Series(rf_full.feature_importances_,   index=X.columns).sort_values(ascending=False).head(25)

fig, axes = plt.subplots(1, 2, figsize=(18, 9))

for ax, fi, title in [(axes[0], fi_lgbm, 'LightGBM'), (axes[1], fi_rf, 'Random Forest')]:
    colors = ['gold' if 'slope' in f else ('tomato' if 'last10' in f or 'max' in f
              else ('mediumpurple' if f in ('lat','lon','win_num') else 'steelblue')) for f in fi.index]
    ax.barh(fi.index[::-1], fi.values[::-1], color=colors[::-1], edgecolor='white')
    ax.set_title(f'{title} — Top 25 Features', fontsize=12)
    ax.set_xlabel('Importance')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(color='mediumpurple', label='Geographic (lat/lon)'),
    Patch(color='gold',         label='Trend (slope)'),
    Patch(color='tomato',       label='Recency (last10/max)'),
    Patch(color='steelblue',    label='Aggregate (mean/std)'),
]
axes[1].legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.suptitle('Feature Importance: LightGBM vs Random Forest', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Feature group breakdown
def categorize(feat):
    if feat in ('lat','lon','win_num'):  return 'Geographic'
    if 'slope'   in feat:               return 'Trend (slope)'
    if 'last10'  in feat:               return 'Recency (last 10d)'
    if 'first10' in feat:               return 'Early (first 10d)'
    if '_std'    in feat:               return 'Variability (std)'
    if '_max'    in feat:               return 'Peak (max)'
    if '_mean'   in feat:               return 'Average (mean)'
    return 'Other'

fi_cats_lgbm = pd.Series(lgbm_full.feature_importances_, index=X.columns)
fi_cats_lgbm = fi_cats_lgbm.groupby(fi_cats_lgbm.index.map(categorize)).sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
fi_cats_lgbm.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('LightGBM Feature Importance by Category', fontsize=12)
ax.set_ylabel('Total Importance')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()

print('Insight: Geographic location dominates, followed by variability and recency features.')
print('Mean values alone are poor predictors — variance/trends carry the signal.')

## 4. ROC Curves — All Models

In [ ]:
from sklearn.model_selection import cross_val_predict

fig, ax = plt.subplots(figsize=(8, 7))
palette = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00','#a65628']

for (name, model), color in zip(models.items(), palette):
    proba = cross_val_predict(model, X, y, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
    fpr, tpr, _ = roc_curve(y, proba)
    auc = roc_auc_score(y, proba)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name}  AUC={auc:.4f}')

ax.plot([0,1],[0,1],'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — 5-Fold CV Predictions', fontsize=13)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 5. Probability Calibration — LightGBM vs Random Forest

In [ ]:
# Calibration curves reveal why LightGBM has higher AUC but worse log-loss
# (its raw probabilities are less reliable / over-confident)
from sklearn.model_selection import cross_val_predict

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models_to_cal = {
    'LightGBM':     models['LightGBM'],
    'RandomForest': models['RandomForest'],
}

for (name, model), ax in zip(models_to_cal.items(), axes):
    proba = cross_val_predict(model, X, y, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
    
    # Raw
    frac_pos, mean_pred = calibration_curve(y, proba, n_bins=15)
    ax.plot(mean_pred, frac_pos, marker='o', label=f'{name} (raw)', color='steelblue', lw=2)
    
    # Platt-scaled
    cal_model = CalibratedClassifierCV(model, cv=3, method='sigmoid')
    proba_cal = cross_val_predict(cal_model, X, y, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
    frac_pos2, mean_pred2 = calibration_curve(y, proba_cal, n_bins=15)
    ll_cal = log_loss(y, proba_cal)
    ax.plot(mean_pred2, frac_pos2, marker='s', label=f'{name} (Platt LL={ll_cal:.4f})', color='tomato', lw=2)
    
    ax.plot([0,1],[0,1],'k--', lw=1, label='Perfect')
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives')
    ax.set_title(f'{name} — Calibration Curve', fontsize=11)
    ax.legend(fontsize=8)

plt.suptitle('Probability Calibration (Platt scaling reduces log-loss)', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Tuned LightGBM — Best Configuration

In [ ]:
# Tune key LightGBM hyperparameters
# Key levers for this dataset:
#   - num_leaves: controls model complexity (default 31, try 31-127)
#   - min_child_samples: regularization against overfitting on small groups
#   - colsample_bytree / subsample: reduce variance
#   - scale_pos_weight: handle class imbalance

configs = [
    {'num_leaves': 31,  'min_child_samples': 20,  'subsample': 0.8, 'colsample_bytree': 0.8,  'label': 'leaves=31'},
    {'num_leaves': 63,  'min_child_samples': 30,  'subsample': 0.8, 'colsample_bytree': 0.8,  'label': 'leaves=63'},
    {'num_leaves': 127, 'min_child_samples': 50,  'subsample': 0.8, 'colsample_bytree': 0.7,  'label': 'leaves=127'},
    {'num_leaves': 63,  'min_child_samples': 30,  'subsample': 0.7, 'colsample_bytree': 0.7,  'label': 'more-dropout'},
    {'num_leaves': 63,  'min_child_samples': 100, 'subsample': 0.8, 'colsample_bytree': 0.8,  'label': 'heavy-reg'},
]

print(f'{"Config":<18} {"ROC-AUC":>9} {"Log-Loss":>10}')
print('-' * 42)
best_auc, best_cfg = 0, None
for cfg in configs:
    label = cfg.pop('label')
    model = lgb.LGBMClassifier(n_estimators=700, learning_rate=0.03,
                                scale_pos_weight=imbalance_ratio,
                                n_jobs=-1, random_state=SEED, verbosity=-1, **cfg)
    scores = cross_validate(model, X, y, cv=cv, scoring=['roc_auc','neg_log_loss'], n_jobs=-1)
    auc = scores['test_roc_auc'].mean()
    ll  = -scores['test_neg_log_loss'].mean()
    print(f'{label:<18} {auc:>9.4f} {ll:>10.4f}')
    if auc > best_auc:
        best_auc, best_cfg = auc, {**cfg, 'label': label}

print(f'\nBest config: {best_cfg} → AUC={best_auc:.4f}')

## 7. Final Model: Calibrated LightGBM

In [ ]:
# Best config from tuning — update if tuning found something better
best_lgbm = lgb.LGBMClassifier(
    n_estimators=700,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=imbalance_ratio,
    n_jobs=-1,
    random_state=SEED,
    verbosity=-1
)

# Wrap with Platt calibration to fix log-loss
final_model = CalibratedClassifierCV(best_lgbm, cv=5, method='sigmoid')

# Final CV evaluation
scores = cross_validate(final_model, X, y, cv=cv,
                        scoring=['roc_auc','neg_log_loss'], n_jobs=-1)
print('=== Final Calibrated LightGBM ===')
print(f"ROC-AUC : {scores['test_roc_auc'].mean():.4f} ± {scores['test_roc_auc'].std():.4f}")
print(f"Log-Loss: {-scores['test_neg_log_loss'].mean():.4f} ± {scores['test_neg_log_loss'].std():.4f}")

In [ ]:
X.columns
test_feat [ X.columns  ]

agg_cols = [c for c in X.columns if c != 'day']
window_df_test = test_feat.groupby('window_id')[agg_cols].mean()

display ( window_df_test)


## 8. Generate Submission

In [ ]:
# Fit on full training data and predict test set
print('Fitting final model on all training data...')
final_model.fit(X, y)

test_proba = final_model.predict_proba(window_df_test)[:, 1]



# # Build submission — one row per window_id
# sub = window_test[['win_num']].copy()
# sub['window_id'] = window_test.index if 'window_id' not in window_test.columns else window_test['window_id']
# # Re-attach window_id
# test_ids = test_feat.drop_duplicates('window_id')[['window_id']].reset_index(drop=True)
# submission = test_ids.copy()
# submission['y'] = test_proba

# print(f'Submission shape: {submission.shape}')
# print(f'Predicted fire rate: {submission["y"].mean():.3f}')
# print(f'Out-of-bounds predictions: {((submission.y > 1) | (submission.y < 0)).sum()}')

# submission.to_csv('/Users/vkdvamshi/gitRepos/ai-ml-learnings/submission.csv', index=False)
# print('Saved: submission_lgbm_calibrated.csv')
# submission.head()

In [94]:
window_df_test['y'] = test_proba

window_df_test.y.to_csv('/Users/vkdvamshi/gitRepos/ai-ml-learnings/submission.csv')

In [ ]:
!kaggle competitions submit spatiotemporal-wildfire-prediction \
    -f /Users/vkdvamshi/gitRepos/ai-ml-learnings/submission.csv \
    -m "LightGBM calibrated, window-level agg+trend features"

## 9. What's Next — Improvement Roadmap

| Priority | Idea | Expected Gain |
|----------|------|---------------|
| **High** | Add percentile features (`p25`, `p75`, `p90` per window) | +0.005–0.01 AUC |
| **High** | Add `erc_last10 / erc_first10` ratio (change factor) | +0.003–0.008 |
| **High** | Stacking: LightGBM + RandomForest meta-learner | +0.003–0.010 |
| **Medium** | Add `max_day` (which day had peak ERC/BI) | +0.002–0.005 |
| **Medium** | Log-transform `pr`, `sph`, `vpd` before aggregation | +0.001–0.003 |
| **Medium** | Train an LGBM per geographic cluster (kmeans on lat/lon) | Variable |
| **Low** | LightGBM early stopping on OOF validation | Avoids overfitting |
| **Low** | Optuna hyperparameter search (100 trials) | +0.002–0.005 |

### Why LightGBM is the right choice
- **Handles mixed feature scales** (no need to standardize)
- **Built-in `scale_pos_weight`** for class imbalance
- **Fast** — 89k windows × 68 features trains in seconds
- **Feature importance** is interpretable and aligns with domain knowledge
- **Platt calibration** fixes its one weakness (overconfident raw probabilities)